My code is broken up into several chunks for more flexibility.
1. Get Torch working with CUDA on my machine
2. Pull out the mean and std. deviation values from the dataset
3. Train the model
4. Test the model

1. Get Torch working with CUDA on my machine

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.version.git_version)

# https://codezup.com/train-first-image-classification-model-with-pytorch/

True
13.0
0fabc3ba44823f257e70ce397d989c8de5e362c1


2. Pull out the std. deviation and mean from the data

In [2]:
# Calculate the mean and std. dev of our dataset
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define the variables
transform = transforms.ToTensor()
dataset = datasets.ImageFolder(root="./train", transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=16)

# Create dummy variables
mean = 0.
std = 0.
num_batches = 0

# Process each image in the data loader
for images, _ in loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    num_batches += batch_samples

mean /= num_batches
std /= num_batches

print(mean, std)

tensor([0.6263, 0.5229, 0.4688]) tensor([0.2797, 0.3395, 0.3697])


3. Train the model

In [3]:
import torch
from PIL import Image
import os
import csv
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

# Create a dataloader for the unlabeled data
class TestImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.image_paths = [
            os.path.join(root, f)
            for f in sorted(os.listdir(root))
            if f.lower().endswith((".jpg"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, os.path.basename(path)

# Data transform (convert to tensor, then normalize to the values from data_preprocessing.py)
transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.626, 0.523, 0.469], std=[0.278, 0.340, 0.370])
])

# Load the training data
train_set = datasets.ImageFolder(root="./train", transform=transform)
train_loader = DataLoader(train_set, batch_size=4, shuffle=True, pin_memory=True, num_workers=16)
# Create the CNN Model
class CNN(nn.Module):
    def __init__(self):
        # Init the super class
        super(CNN, self).__init__()
        
        # First layer: convolution. RGB in, 6 features out. Kernel size 5
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        
        # Second layer: maxpool. 2x2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Third layer: convolution: 6 channel in, 16 features out, kernel size 5
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)

        self.conv3 = nn.Conv2d(in_channels=16, out_channels=64, kernel_size=5)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=256, kernel_size=5)
        
        # Fourth layer, Linear. 16x5x5 in, 120 out
        self.fc1 = nn.Linear(in_features=256*2*2, out_features=240)
        
        # Fifth layer, Linear. 120 in, 84 out
        self.fc2 = nn.Linear(240, 120)

        self.fc3 = nn.Linear(120, 84)
        
        # Sixth layer, output layer, Linear. 84 in, 7 out
        self.fc4 = nn.Linear(84, 7)

    def forward(self, x):
        # Convolution and pooling, twice. Using ReLU
        x = self.pool(torch.relu(self.conv1(x)))  # 100 -> 48
        x = self.pool(torch.relu(self.conv2(x)))  # 48 -> 22
        x = self.pool(torch.relu(self.conv3(x)))  # 22 -> 9
        x = self.pool(torch.relu(self.conv4(x)))  # 9 -> 2
        
        # Reshape the tensor to flatten in
        x = x.view(x.size(0), -1)
        
        # Basic ReLU activations
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        
        # Final linear transform
        x = self.fc4(x)
        return x

# Training loop
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
net = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

# Train
for epoch in range(5):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print('[Epoch %d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / (i + 1)))

print('Finished Training!')

# Save the model
torch.save(net.state_dict(), "cnn_model_state.pth")
print("Saved Model!")


Using device: cuda
[Epoch 1,  8874] loss: 0.469
[Epoch 2,  8874] loss: 0.166
[Epoch 3,  8874] loss: 0.134
[Epoch 4,  8874] loss: 0.121
[Epoch 5,  8874] loss: 0.103
Finished Training!
Saved Model!


5. Test model

In [4]:
import torch
from PIL import Image
import os
import csv
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn

# Test dataset
class TestImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.image_paths = [
            os.path.join(root, f)
            for f in sorted(os.listdir(root))
            if f.lower().endswith(".jpg")
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, os.path.basename(path)

# Transform
transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.626, 0.523, 0.469], std=[0.278, 0.340, 0.370])
])

test_set = TestImageDataset(root="./test", transform=transform)
test_loader = DataLoader(test_set, batch_size=4, shuffle=False)

# Create the CNN Model
class CNN(nn.Module):
    def __init__(self):
        # Init the super class
        super(CNN, self).__init__()
        
        # First layer: convolution. RGB in, 6 features out. Kernel size 5
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        
        # Second layer: maxpool. 2x2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Third layer: convolution: 6 channel in, 16 features out, kernel size 5
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)

        self.conv3 = nn.Conv2d(in_channels=16, out_channels=64, kernel_size=5)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=256, kernel_size=5)
        
        # Fourth layer, Linear. 16x5x5 in, 120 out
        self.fc1 = nn.Linear(in_features=256*2*2, out_features=240)
        
        # Fifth layer, Linear. 120 in, 84 out
        self.fc2 = nn.Linear(240, 120)

        self.fc3 = nn.Linear(120, 84)
        
        # Sixth layer, output layer, Linear. 84 in, 7 out
        self.fc4 = nn.Linear(84, 7)

    def forward(self, x):
        # Convolution and pooling, twice. Using ReLU
        x = self.pool(torch.relu(self.conv1(x)))  # 100 -> 48
        x = self.pool(torch.relu(self.conv2(x)))  # 48 -> 22
        x = self.pool(torch.relu(self.conv3(x)))  # 22 -> 9
        x = self.pool(torch.relu(self.conv4(x)))  # 9 -> 2
        
        # Reshape the tensor to flatten in
        x = x.view(x.size(0), -1)
        
        # Basic ReLU activations
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        
        # Final linear transform
        x = self.fc4(x)
        return x


# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = CNN().to(device)
net.load_state_dict(torch.load("cnn_model_state.pth", map_location=device))
net.eval()

# Make predictions
predictions = []
with torch.no_grad():
    for images, filenames in test_loader:
        images = images.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        for fname, pred in zip(filenames, predicted):
            predictions.append((fname.split('.')[0], pred.item()))

# Save CSV
csv_path = "test_predictions.csv"
with open(csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "PredictedClass"])
    for idx, pred_class in predictions:
        print(idx, pred_class)
        writer.writerow([idx, pred_class])

print(f"Predictions saved to {csv_path}")


00000 0
00001 0
00002 0
00003 0
00004 1
00005 0
00006 2
00007 1
00008 2
00009 0
00010 0
00011 2
00012 2
00013 1
00014 2
00015 5
00016 0
00017 2
00018 0
00019 4
00020 2
00021 0
00022 5
00023 0
00024 0
00025 2
00026 0
00027 2
00028 0
00029 2
00030 5
00031 2
00032 2
00033 2
00034 0
00035 0
00036 5
00037 1
00038 1
00039 0
00040 0
00041 2
00042 0
00043 5
00044 0
00045 5
00046 0
00047 2
00048 0
00049 4
00050 0
00051 0
00052 2
00053 2
00054 2
00055 0
00056 6
00057 4
00058 2
00059 0
00060 0
00061 0
00062 1
00063 5
00064 0
00065 0
00066 0
00067 2
00068 6
00069 0
00070 5
00071 0
00072 0
00073 0
00074 0
00075 2
00076 3
00077 0
00078 0
00079 0
00080 0
00081 0
00082 1
00083 2
00084 0
00085 1
00086 3
00087 0
00088 0
00089 2
00090 4
00091 0
00092 2
00093 0
00094 0
00095 2
00096 1
00097 0
00098 0
00099 2
00100 5
00101 0
00102 2
00103 0
00104 4
00105 4
00106 5
00107 0
00108 2
00109 0
00110 0
00111 0
00112 1
00113 0
00114 3
00115 0
00116 5
00117 4
00118 0
00119 4
00120 3
00121 5
00122 0
00123 0
00124 0
